In [8]:
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path

initialDB_path = Path('../data/initialDB_sequences.csv')
initialDB = pd.read_csv(initialDB_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len'
])

df = initialDB.copy()
df.head()

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494


In [ ]:
def gc_content(seq):
    if not isinstance(seq, str):
        return np.nan
    seq = seq.strip().upper()
    if not seq:
        return np.nan
    gc = seq.count('G') + seq.count('C')
    return gc / len(seq)

df = initialDB.copy()
df['5_UTR_GC'] = df['5_UTR'].fillna('').map(gc_content)
df['CDS_GC'] = df['CDS'].fillna('').map(gc_content)
df['3_UTR_GC'] = df['3_UTR'].fillna('').map(gc_content)
df['global_GC'] = df[['5_UTR', 'CDS', '3_UTR']].fillna('').agg(''.join, axis=1).map(gc_content)

df.head()

In [ ]:
motifs = {
"ATTTA": r"A[TU]{3}A",
"WTTTW": r"[ACGTU][TU]{3}[ACGTU]",
"WWTTTWW": r"[ACGTU]{2}[TU]{3}[ACGTU]{2}",
"WWWTTTWWW": r"[ACGTU]{3}[TU]{3}[ACGTU]{3}",
"WWWWTTTWWWW": r"[ACGTU]{4}[TU]{3}[ACGTU]{4}",
"WWWWWTTTWWWWW": r"[ACGTU]{5}[TU]{3}[ACGTU]{5}",
"TTTGTTT": r"[TU]{3}G[TU]{3}",
"GTTTG": r"G[TU]{3}G",
"AWTAAA": r"A[ACGTU][TU]AAA"
}

def scan_motifs(df, sequence_col, count_prefix):
    """
    Scans a specific column for motifs and adds a total count column.
    Returns a detailed breakdown DataFrame for those specific motifs.
    """
    total_col_name = f"{count_prefix}_AREs"
    df[total_col_name] = 0
    
    motif_counts_dict = {
        'gene_id': df['gene_id'],
        'transcript_id': df['transcript_id'],
        f'{count_prefix}_sequence': df[sequence_col]
    }
    
    for name, pattern in motifs.items():
        counts = df[sequence_col].str.findall(pattern, flags=re.IGNORECASE).str.len().fillna(0).astype(int)
        
        df[total_col_name] += counts
        
        motif_counts_dict[f"{count_prefix}_{name}_count"] = counts
    
    detailed_df = pd.DataFrame(motif_counts_dict)
    return df, detailed_df

df, AREs_5_df = scan_motifs(df, '5_UTR', '5')

df, AREs_3_df = scan_motifs(df, '3_UTR', '3')

df.head(10)

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,3_UTR_GC,global_GC,5_AREs,3_AREs
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.400000,0.426380,NaN,0.424855,5,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.809430,0.701422,NaN,0.719500,0,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.625000,0.603471,0.572874,0.598114,0,45
5,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,110,1926,0,0.745455,0.663032,NaN,0.667485,0,0
6,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,50,1833,0,0.640000,0.667758,NaN,0.667021,0,0
7,ENSG00000187642.11,ENST00000433179.4,AGACGCCCAGCTCGGCCGCCGGGACCCAGGGCATGGATGGAGCCCC...,ATGGAAAATTTCCAGTACAGCGTCCAGCTGAGTGACCAGGACTGGG...,TAGGAGCCAGGCCCGGGCCAGGGAGATGCAGGATGAGGAGACGACC...,173,2370,977,0.705202,0.660759,0.634596,0.655682,0,27
8,ENSG00000188290.12,ENST00000304952.11,GCGGGCCTGGAGCCGGGATCCGCCCTAGGGGCTCGGATCGCCGCGC...,ATGGCCGCAGACACGCCGGGGAAACCGAGCGCCTCGCCGATGGCAG...,TGAGGCTGTGGCCCTGAGACTGCATCGGAGGCGGCGCCCCGTTCTA...,124,663,98,0.838710,0.743590,0.571429,0.737853,0,11
9,ENSG00000187608.11,ENST00000649529.1,GGCGGCTGAGAGGCAGCGAACTCATCTTTGCCAGTACAGGAGCTTG...,ATGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGCAACGAATTCC...,NaN,77,495,0,0.649351,0.658586,NaN,0.657343,5,0


In [ ]:
def scan_uorf(df, utr5_col):
    """
    Scans 5' UTR for:
    1. uAUGs: Any AUG in the 5' UTR
    2. uORF: AUG in frame with the canonical start codon 
    """

    def classify_utr_motifs(seq):
        if pd.isna(seq) or len(seq) < 3:
            return 0, 0
        
        seq = seq.upper()
        # 1. uAUG_count: Find all ATG positions
        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]
        uaug_count = len(atg_indices)
        
        # 2. uORF: ATG in frame with the canonical start codon
        # The canonical start codon begins immediately after the 5' UTR.
        # Therefore, an AUG is in-frame if the distance from AUG to the 
        # end of the sequence is a multiple of 3.
        uorf_count = 0
        seq_len = len(seq)
        for start_idx in atg_indices:
            distance_to_end = seq_len - start_idx
            if distance_to_end % 3 == 0:
                uorf_count += 1
                
        return uaug_count, uorf_count

    results = df[utr5_col].apply(classify_utr_motifs)
    df[['uAUG_count', 'uORF_in_frame']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

def scan_dorf(df, utr3_col):
    """
    Scans 3' UTR for dORFs: 
    Downstream ORFs starting with AUG in the 3' UTR.
    Returns count of dORFs that find a stop codon vs. those that don't (overlap/truncated).
    """

    def classify_dorf_motifs(seq):
        if pd.isna(seq) or len(seq) < 3:
            return 0, 0
        
        seq = seq.upper()
        atg_indices = [m.start() for m in re.finditer(r'(?=ATG)', seq)]
        
        non_overlap = 0 # Finding a stop codon within the 3' UTR
        overlap = 0     # Running off the end of the 3' UTR without a stop
        
        stops = {'TAA', 'TAG', 'TGA'}

        for start_idx in atg_indices:
            remaining_seq = seq[start_idx:]
            found_stop = False
            
            # Check every 3 bases from the current ATG
            for i in range(0, len(remaining_seq) - 2, 3):
                codon = remaining_seq[i:i+3]
                if codon in stops:
                    found_stop = True
                    break
            
            if found_stop:
                non_overlap += 1
            else:
                overlap += 1
                
        return non_overlap, overlap

    results = df[utr3_col].apply(classify_dorf_motifs)
    df[['dORF_with_stop', 'dORF_truncated']] = pd.DataFrame(
        results.tolist(), index=df.index
    )
    return df

df = scan_uorf(df, '5_UTR')
df = scan_dorf(df, '3_UTR')

df.head(10)

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,3_UTR_GC,global_GC,5_AREs,3_AREs,uAUG_count,uORF_non_overlap,uORF_overlap,dORF_non_overlap,dORF_overlap
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.400000,0.426380,NaN,0.424855,5,0,1,0,1,0,0
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0,0,0,0,0,0
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,0.000000,0.460064,0,0,0,0,0,0,0
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.809430,0.701422,NaN,0.719500,0,0,1,1,0,0,0
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.625000,0.603471,0.572874,0.598114,0,45,0,0,0,1,0
5,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,110,1926,0,0.745455,0.663032,NaN,0.667485,0,0,0,0,0,0,0
6,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,50,1833,0,0.640000,0.667758,NaN,0.667021,0,0,0,0,0,0,0
7,ENSG00000187642.11,ENST00000433179.4,AGACGCCCAGCTCGGCCGCCGGGACCCAGGGCATGGATGGAGCCCC...,ATGGAAAATTTCCAGTACAGCGTCCAGCTGAGTGACCAGGACTGGG...,TAGGAGCCAGGCCCGGGCCAGGGAGATGCAGGATGAGGAGACGACC...,173,2370,977,0.705202,0.660759,0.634596,0.655682,0,27,4,3,1,8,1
8,ENSG00000188290.12,ENST00000304952.11,GCGGGCCTGGAGCCGGGATCCGCCCTAGGGGCTCGGATCGCCGCGC...,ATGGCCGCAGACACGCCGGGGAAACCGAGCGCCTCGCCGATGGCAG...,TGAGGCTGTGGCCCTGAGACTGCATCGGAGGCGGCGCCCCGTTCTA...,124,663,98,0.838710,0.743590,0.571429,0.737853,0,11,0,0,0,0,0
9,ENSG00000187608.11,ENST00000649529.1,GGCGGCTGAGAGGCAGCGAACTCATCTTTGCCAGTACAGGAGCTTG...,ATGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGCAACGAATTCC...,NaN,77,495,0,0.649351,0.658586,NaN,0.657343,5,0,0,0,0,0,0


In [ ]:
IRES_path = Path('../data/utr_ire_scan_results.csv')
IRES_db = pd.read_csv(IRES_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_UTR_len', 'CDS_len', '3_UTR_len', 'IRES count', 'IRES IDs'
])


cols_to_keep = [col for col in IRES_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, IRES_db[cols_to_keep], on='transcript_id', how='left')

df.head(10)

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,...,global_GC,5_AREs,3_AREs,uAUG_count,uORF_non_overlap,uORF_overlap,dORF_non_overlap,dORF_overlap,IRES count,IRES IDs
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.400000,0.426380,...,0.424855,5,0,1,0,1,0,0,0,NaN
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,...,0.460064,0,0,0,0,0,0,0,0,NaN
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,...,0.460064,0,0,0,0,0,0,0,0,NaN
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.809430,0.701422,...,0.719500,0,0,1,1,0,0,0,0,NaN
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.625000,0.603471,...,0.598114,0,45,0,0,0,1,0,0,NaN
5,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,110,1926,0,0.745455,0.663032,...,0.667485,0,0,0,0,0,0,0,0,NaN
6,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,50,1833,0,0.640000,0.667758,...,0.667021,0,0,0,0,0,0,0,0,NaN
7,ENSG00000187642.11,ENST00000433179.4,AGACGCCCAGCTCGGCCGCCGGGACCCAGGGCATGGATGGAGCCCC...,ATGGAAAATTTCCAGTACAGCGTCCAGCTGAGTGACCAGGACTGGG...,TAGGAGCCAGGCCCGGGCCAGGGAGATGCAGGATGAGGAGACGACC...,173,2370,977,0.705202,0.660759,...,0.655682,0,27,4,3,1,8,1,0,NaN
8,ENSG00000188290.12,ENST00000304952.11,GCGGGCCTGGAGCCGGGATCCGCCCTAGGGGCTCGGATCGCCGCGC...,ATGGCCGCAGACACGCCGGGGAAACCGAGCGCCTCGCCGATGGCAG...,TGAGGCTGTGGCCCTGAGACTGCATCGGAGGCGGCGCCCCGTTCTA...,124,663,98,0.838710,0.743590,...,0.737853,0,11,0,0,0,0,0,0,NaN
9,ENSG00000187608.11,ENST00000649529.1,GGCGGCTGAGAGGCAGCGAACTCATCTTTGCCAGTACAGGAGCTTG...,ATGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGCAACGAATTCC...,NaN,77,495,0,0.649351,0.658586,...,0.657343,5,0,0,0,0,0,0,0,NaN


In [ ]:
mfe_path = Path('../data/mfe_results.csv')
mfe_db = pd.read_csv(mfe_path, sep=',', header=0, comment='#', names=[
    'gene_id', 'transcript_id', '5_UTR', 'CDS', '3_UTR', '5_len', 'CDS_len', '3_len', '5_UTR_MFE', 'CDS_MFE', '3_UTR_MFE'
])
### Note: MFE values of NaN were not able to be folded by RNAfold either because the sequence is length 0 or it's too long (over 100k nt) for RNAfold to handle.

cols_to_keep = [col for col in mfe_db.columns if col == 'transcript_id' or col not in df.columns]

df = pd.merge(df, mfe_db[cols_to_keep], on='transcript_id', how='left')

df.head(10)

,gene_id,transcript_id,5_UTR,CDS,3_UTR,5_UTR_len,CDS_len,3_UTR_len,5_UTR_GC,CDS_GC,...,uORF_overlap,dORF_non_overlap,dORF_overlap,IRES count,IRES IDs,5_len,3_len,5_UTR_MFE,CDS_MFE,3_UTR_MFE
0,ENSG00000186092.7,ENST00000641515.2,CCCAGATCTCTTCAGTTTTTATGCCTCATTCTGTGAAAATTGCTGT...,ATGAAGAAGGTAACTGCAGAGGCTATTTCCTGGAATGAATCAACGA...,NaN,60,978,0,0.400000,0.426380,...,1,0,0,0,NaN,60,0,-9.5,-260.2,NaN
1,ENSG00000284733.2,ENST00000426406.4,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,...,0,0,0,0,NaN,0,3,NaN,-256.7,0.0
2,ENSG00000284662.2,ENST00000332831.5,NaN,ATGGATGGAGAGAATCACTCAGTGGTATCTGAGTTTTTGTTTCTGG...,TAA,0,936,3,NaN,0.461538,...,0,0,0,0,NaN,0,3,NaN,-256.7,0.0
3,ENSG00000187634.14,ENST00000616016.5,GGCGGCGGAGTCTCCCAAGTCCCCGCCGGGCGGGCGCGCGCCAGTG...,ATGCCGGCGGTCAAGAAGGAGTTCCCGGGCCGCGAGGACCTGGCCC...,NaN,509,2532,0,0.809430,0.701422,...,0,0,0,0,NaN,509,0,-321.0,-1251.6,NaN
4,ENSG00000188976.12,ENST00000327044.7,GCTTCGGGTTGGTGTC,ATGGCAGCTGCGGGGAGCCGCAAGAGGCGCCTGGCGGAGCTGACGG...,TGAGGCAGCCCATCTGGGGGGCCTGTAGGGGCTGCCGGGCTGGTGG...,16,2247,494,0.625000,0.603471,...,0,1,0,0,NaN,16,494,0.0,-943.8,-209.8
5,ENSG00000187961.16,ENST00000338591.8,GGGAGTGAGCGACACAGAGCGGGCCGCCACCGCCGAGCAGCCCTCC...,ATGCAGCCCCGCAGCGAGCGCCCGGCCGGCAGGACGCAGAGCCCGG...,NaN,110,1926,0,0.745455,0.663032,...,0,0,0,0,NaN,110,0,-45.2,-879.6,NaN
6,ENSG00000187583.12,ENST00000379410.8,AGGAGGCTGTGGACAGGGACCCAGACTTGCCGACCTGTACGACTCT...,ATGGGGAACAGCCACTGTGTCCCTCAGGCCCCCAGGAGGCTCCGGG...,NaN,50,1833,0,0.640000,0.667758,...,0,0,0,0,NaN,50,0,-12.9,-821.6,NaN
7,ENSG00000187642.11,ENST00000433179.4,AGACGCCCAGCTCGGCCGCCGGGACCCAGGGCATGGATGGAGCCCC...,ATGGAAAATTTCCAGTACAGCGTCCAGCTGAGTGACCAGGACTGGG...,TAGGAGCCAGGCCCGGGCCAGGGAGATGCAGGATGAGGAGACGACC...,173,2370,977,0.705202,0.660759,...,1,8,1,0,NaN,173,977,-79.2,-1128.0,-457.1
8,ENSG00000188290.12,ENST00000304952.11,GCGGGCCTGGAGCCGGGATCCGCCCTAGGGGCTCGGATCGCCGCGC...,ATGGCCGCAGACACGCCGGGGAAACCGAGCGCCTCGCCGATGGCAG...,TGAGGCTGTGGCCCTGAGACTGCATCGGAGGCGGCGCCCCGTTCTA...,124,663,98,0.838710,0.743590,...,0,0,0,0,NaN,124,98,-67.6,-347.0,-32.7
9,ENSG00000187608.11,ENST00000649529.1,GGCGGCTGAGAGGCAGCGAACTCATCTTTGCCAGTACAGGAGCTTG...,ATGGGCTGGGACCTGACGGTGAAGATGCTGGCGGGCAACGAATTCC...,NaN,77,495,0,0.649351,0.658586,...,0,0,0,0,NaN,77,0,-23.7,-223.3,NaN
